In [1]:
#prep relevent packages 
from pathlib import Path
import pandas as pd
import re
import pdfplumber

In [2]:
# file path to import data
root_fldr = Path.cwd().parent
raw_fldr = root_fldr / 'data' / 'raw' 
processed_fldr = root_fldr / 'data' / 'processed'

# import cfp residential risk pdf 
risk_exposure = raw_fldr / 'cfp_data' / 'CFP-5-yr-TIV-County-FY26-All-261020.pdf'

In [ ]:
rows = []  # reset each run so reruns don't duplicate rows

#help provided with some AI, particularly with county row check to get unified, scraped dataframe
with pdfplumber.open(risk_exposure) as pdf:
    for page in pdf.pages:
        for line in page.extract_text().split('\n'):
            line = line.replace('$', '')
            m = re.match(r'([A-Za-z .]+?)\s+(-?\d+%.*)$', line)
            if m is None or m.group(1).strip() == 'Total':
                continue
            county, rest = m.group(1).strip(), m.group(2)
            pairs = re.findall(r'(-?\d+)%\s+([\d ,]+?)(?=\s+-?\d+%|\s*$)', rest)
            if len(pairs) == 5:
                values = []
                for growth, exposure in pairs:
                    values += [growth, exposure.replace(' ', '').replace(',', '')]
                rows.append([county] + values)

cols = ['county',
        'growth_2026', 'exposure_2026',
        'growth_2025', 'exposure_2025',
        'growth_2024', 'exposure_2024',
        'growth_2023', 'exposure_2023',
        'growth_2022', 'exposure_2022']
risk_exposure_df = pd.DataFrame(rows, columns=cols)
risk_exposure_df[cols[1:]] = risk_exposure_df[cols[1:]].apply(pd.to_numeric)



58
760293459869


In [4]:
#check dataframe to see if it can be read  
risk_exposure_df.head()

,county,growth_2026,exposure_2026,growth_2025,exposure_2025,growth_2024,exposure_2024,growth_2023,exposure_2023,growth_2022,exposure_2022
0,Alameda,11,20620263185,89,18552360134,139,9822579031,35,4112147801,41,3047740652
1,Alpine,-45,181267207,20,330839876,24,276712560,24,222282425,34,179049160
2,Amador,1,5170423411,17,5112500719,24,4365314159,26,3526367836,22,2808434094
3,Butte,-4,7012819946,20,7280278840,51,6052068909,62,4007419450,25,2472656045
4,Calaveras,4,9358705220,16,8987677408,19,7730480439,31,6505003809,26,4970383068


In [8]:
#converting df into appropriate variable types 

risk_exposure_clean = (
    risk_exposure_df
    .set_index('county') 
    .reset_index()
    .assign( #converting into percentages 
        growth_2026 = lambda d: d['growth_2026'] / 100,
        growth_2025 = lambda d: d['growth_2025'] / 100,
        growth_2024 = lambda d: d['growth_2024'] / 100,
        growth_2023 = lambda d: d['growth_2023'] / 100,
        growth_2022 = lambda d: d['growth_2022'] / 100
    )
    .astype({
        'county': 'string',
        'growth_2026': 'float64',
        'exposure_2026': 'Int64',
        'growth_2025': 'float64',
        'exposure_2025': 'Int64',
        'growth_2024': 'float64',
        'exposure_2024': 'Int64',
        'growth_2023': 'float64',
        'exposure_2023': 'Int64',
        'growth_2022': 'float64',
        'exposure_2022': 'Int64'
    })
)

risk_exposure_clean.head()


,county,growth_2026,exposure_2026,growth_2025,exposure_2025,growth_2024,exposure_2024,growth_2023,exposure_2023,growth_2022,exposure_2022
0,Alameda,0.11,20620263185,0.89,18552360134,1.39,9822579031,0.35,4112147801,0.41,3047740652
1,Alpine,-0.45,181267207,0.20,330839876,0.24,276712560,0.24,222282425,0.34,179049160
2,Amador,0.01,5170423411,0.17,5112500719,0.24,4365314159,0.26,3526367836,0.22,2808434094
3,Butte,-0.04,7012819946,0.20,7280278840,0.51,6052068909,0.62,4007419450,0.25,2472656045
4,Calaveras,0.04,9358705220,0.16,8987677408,0.19,7730480439,0.31,6505003809,0.26,4970383068


In [9]:
risk_exposure_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   county         58 non-null     string 
 1   growth_2026    58 non-null     float64
 2   exposure_2026  58 non-null     Int64  
 3   growth_2025    58 non-null     float64
 4   exposure_2025  58 non-null     Int64  
 5   growth_2024    58 non-null     float64
 6   exposure_2024  58 non-null     Int64  
 7   growth_2023    58 non-null     float64
 8   exposure_2023  58 non-null     Int64  
 9   growth_2022    58 non-null     float64
 10  exposure_2022  58 non-null     Int64  
dtypes: Int64(5), float64(5), string(1)
memory usage: 5.8 KB


In [ ]:
#check to see if numbers from pull are aligned with pdf 
# Totals from pdf
# 2026 exposure = $760,293,459,869
# 2025 exposure = $693,964,308,706
# 2024 exposure = $458,023,820,951
total_2026_exposure = risk_exposure_clean['exposure_2026'].sum()
total_2025_exposure = risk_exposure_clean['exposure_2025'].sum()
total_2024_exposure = risk_exposure_clean['exposure_2024'].sum()

print('2026 exposure:', total_2026_exposure)
print('2025 exposure:', total_2025_exposure)
print('2024 exposure:', total_2024_exposure)


2025 exposure: 645115692650
2024 exposure: 431066598171
2023 exposure: 270510184565


In [11]:
#Enables a quick review of the pdf to ensure specific values in risk_exposure_clean align with the pdf 
county_check = 'Alameda'
year = 2026
exposure_check = risk_exposure_clean.loc[risk_exposure_clean['county']==county_check, f'exposure_{year}'].iloc[0]

print('County:', county_check)
print('Year:', year)
print('Total exposure:', exposure_check)


County: Alameda
Year: 2026
Total exposure: 20620263185


In [12]:
#converting to processed folder csv 
risk_exposure_clean.to_csv(processed_fldr / 'cfp_risk_exposure_cleaned.csv')
